In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/movie-recommender/data/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
ratings = pd.read_csv("/content/drive/MyDrive/movie-recommender-clean/movie-recommender-clean/data/ratings.csv")
movies = pd.read_csv("/content/drive/MyDrive/movie-recommender-clean/movie-recommender-clean/data/movies.csv")
links = pd.read_csv("/content/drive/MyDrive/movie-recommender-clean/movie-recommender-clean/data/links.csv")
print(ratings.shape)
print(ratings.dtypes)
print(ratings.isnull().sum())
ratings.head()

(100836, 4)
userId         int64
movieId        int64
rating       float64
timestamp      int64
dtype: object
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
n_users = ratings['userId'].nunique() # how many unique users
n_movies = ratings['movieId'].nunique() # how many unique movies
n_ratings = len(ratings)
sparsity = 1 - (n_ratings / (n_users * n_movies))
print(f'Users: {n_users:,}')
print(f'Movies: {n_movies:,}')
print(f'Ratings: {n_ratings:,}')
print(f'Sparsity: {sparsity:.4%}') # expect ~98% sparse
# Rating distribution
print(ratings['rating'].describe())
print(ratings['rating'].value_counts().sort_index())

Users: 610
Movies: 9,724
Ratings: 100,836
Sparsity: 98.3000%
count    100836.000000
mean          3.501557
std           1.042529
min           0.500000
25%           3.000000
50%           3.500000
75%           4.000000
max           5.000000
Name: rating, dtype: float64
rating
0.5     1370
1.0     2811
1.5     1791
2.0     7551
2.5     5550
3.0    20047
3.5    13136
4.0    26818
4.5     8551
5.0    13211
Name: count, dtype: int64


In [4]:
# Ratings per user
ratings_per_user = ratings.groupby('userId')['rating'].count()
print(f'Avg ratings per user: {ratings_per_user.mean():.1f}')
print(f'Min: {ratings_per_user.min()} Max: {ratings_per_user.max()}')
# Ratings per movie
ratings_per_movie = ratings.groupby('movieId')['rating'].count()
print(f'Avg ratings per movie: {ratings_per_movie.mean():.1f}')
# Most-rated movies
top_rated = (ratings.groupby('movieId')['rating']
 .count()
 .reset_index()
 .merge(movies, on='movieId')
 .sort_values('rating', ascending=False)
 .head(10))
print(top_rated[['title', 'rating']])

Avg ratings per user: 165.3
Min: 20 Max: 2698
Avg ratings per movie: 10.4
                                          title  rating
314                         Forrest Gump (1994)     329
277            Shawshank Redemption, The (1994)     317
257                         Pulp Fiction (1994)     307
510            Silence of the Lambs, The (1991)     279
1938                         Matrix, The (1999)     278
224   Star Wars: Episode IV - A New Hope (1977)     251
418                        Jurassic Park (1993)     238
97                            Braveheart (1995)     237
507           Terminator 2: Judgment Day (1991)     224
461                     Schindler's List (1993)     220


In [7]:
import pandas as pd
import numpy as np
import joblib, os
from sklearn.model_selection import train_test_split
from google.colab import drive

drive.mount('/content/drive')
BASE       = '/content/drive/MyDrive/movie-recommender-clean/movie-recommender-clean'
DATA_DIR   = f'{BASE}/data/'
MODEL_PATH = f'{BASE}/model/saved/svd_model.pkl'
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

# ── 1. Load ──────────────────────────────────────────────
print('Loading ratings...')
ratings = pd.read_csv(f'{DATA_DIR}ratings.csv')
print(f"{len(ratings):,} ratings | {ratings['userId'].nunique():,} users")

# ── 2. Encode IDs to 0-based matrix indices ──────────────
user_ids  = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()

user_to_idx  = {uid: i for i, uid in enumerate(user_ids)}
movie_to_idx = {mid: i for i, mid in enumerate(movie_ids)}
idx_to_movie = {i: mid for mid, i in movie_to_idx.items()}

n_users  = len(user_ids)
n_movies = len(movie_ids)
print(f'Matrix shape: {n_users} x {n_movies}')

# ── 3. Build Rating Matrix ────────────────────────────────
# Rows = users, Columns = movies, 0 = unrated
R = np.zeros((n_users, n_movies), dtype=np.float32)
for row in ratings.itertuples():
    R[user_to_idx[row.userId], movie_to_idx[row.movieId]] = row.rating

# ── 4. Mean-center (per user) ─────────────────────────────
# Subtract each user's average so SVD learns deviations
user_means = np.zeros(n_users)
for u in range(n_users):
    rated = R[u, R[u] != 0]
    user_means[u] = rated.mean() if len(rated) > 0 else 0.0

R_centered = R.copy()
for u in range(n_users):
    mask = R_centered[u] != 0
    R_centered[u, mask] -= user_means[u]

# ── 5. SVD Decomposition ─────────────────────────────────
k = 50   # latent factors — increase for better accuracy
print('\nRunning SVD (takes ~1-2 min)...')
U, sigma, Vt = np.linalg.svd(R_centered, full_matrices=False)

# Keep top-k factors only
U_k     = U[:, :k]
sigma_k = np.diag(sigma[:k])
Vt_k    = Vt[:k, :]

# Reconstructed matrix of predicted deviations
R_pred = np.dot(np.dot(U_k, sigma_k), Vt_k)
print('SVD complete.')

# ── 6. Evaluate ───────────────────────────────────────────
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

def predict_rating(user_id, movie_id):
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return float(ratings['rating'].mean())
    u    = user_to_idx[user_id]
    m    = movie_to_idx[movie_id]
    pred = R_pred[u, m] + user_means[u]
    return float(np.clip(pred, 0.5, 5.0))

preds   = [predict_rating(r.userId, r.movieId) for r in test_data.itertuples()]
actuals = test_data['rating'].values
rmse    = np.sqrt(np.mean((actuals - np.array(preds))**2))
mae     = np.mean(np.abs(actuals - np.array(preds)))
print(f'\nRMSE: {rmse:.4f}  |  MAE: {mae:.4f}')

# ── 7. Save ───────────────────────────────────────────────
joblib.dump({
    'R_pred':       R_pred,
    'user_means':   user_means,
    'user_to_idx':  user_to_idx,
    'movie_to_idx': movie_to_idx,
    'idx_to_movie': idx_to_movie,
    'ratings':      ratings,
    'global_mean':  float(ratings['rating'].mean()),
}, MODEL_PATH)
print(f'\nSaved to {MODEL_PATH}')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading ratings...
100,836 ratings | 610 users
Matrix shape: 610 x 9724

Running SVD (takes ~1-2 min)...
SVD complete.

RMSE: 0.6220  |  MAE: 0.3984

Saved to /content/drive/MyDrive/movie-recommender-clean/movie-recommender-clean/model/saved/svd_model.pkl


In [8]:
%%writefile /content/drive/MyDrive/movie-recommender-clean/movie-recommender-clean/model/recommender.py
# model/recommender.py
import pandas as pd
import numpy as np
import joblib, requests, os
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

load_dotenv()
OMDB_API_KEY = os.getenv('OMDB_API_KEY')

BASE       = '/content/drive/MyDrive/movie-recommender-clean/movie-recommender-clean'
DATA_DIR   = f'{BASE}/data/'
MODEL_PATH = f'{BASE}/model/saved/svd_model.pkl'

# ── Load SVD artifacts (once at import) ──────────────────
_cache       = joblib.load(MODEL_PATH)
R_pred       = _cache['R_pred']
user_means   = _cache['user_means']
user_to_idx  = _cache['user_to_idx']
movie_to_idx = _cache['movie_to_idx']
ratings_df   = _cache['ratings']
global_mean  = _cache['global_mean']

movies_df = pd.read_csv(f'{DATA_DIR}movies.csv')
links_df  = pd.read_csv(f'{DATA_DIR}links.csv')
tags_df   = pd.read_csv(f'{DATA_DIR}tags.csv')

# ── Build TF-IDF content matrix (once at import) ─────────
tags_agg = tags_df.groupby('movieId')['tag'].apply(' '.join).reset_index()
movies_e = movies_df.merge(tags_agg, on='movieId', how='left')
movies_e['tag']     = movies_e['tag'].fillna('')
movies_e['content'] = movies_e['genres'].str.replace('|',' ') + ' ' + movies_e['tag']

tfidf        = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies_e['content'])
cosine_sim   = cosine_similarity(tfidf_matrix, tfidf_matrix)
mid_to_idx   = pd.Series(movies_e.index, index=movies_e['movieId'])

# ── CF prediction ─────────────────────────────────────────
def predict_rating(user_id, movie_id) -> float:
    if user_id not in user_to_idx or movie_id not in movie_to_idx:
        return global_mean
    u    = user_to_idx[user_id]
    m    = movie_to_idx[movie_id]
    pred = R_pred[u, m] + user_means[u]
    return float(np.clip(pred, 0.5, 5.0))

# ── OMDB poster fetch ─────────────────────────────────────
def get_poster_url(imdb_id) -> str | None:
    if not imdb_id or pd.isna(imdb_id):
        return None
    try:
        imdb_str = f'tt{int(imdb_id):07d}'  # e.g. tt0114709
        resp = requests.get(
            'http://www.omdbapi.com/',
            params={'i': imdb_str, 'apikey': OMDB_API_KEY},
            timeout=5
        )
        poster = resp.json().get('Poster')
        return poster if poster and poster != 'N/A' else None
    except Exception:
        return None

# ── Hybrid recommendation ─────────────────────────────────
def get_hybrid_recommendations(user_id: int, top_n: int = 10) -> list[dict]:
    ALPHA = 0.7   # 70% CF, 30% CB
    user_ratings = ratings_df[ratings_df['userId'] == user_id]
    rated_ids    = set(user_ratings['movieId'])
    candidates   = movies_df[~movies_df['movieId'].isin(rated_ids)].copy()

    if candidates.empty:
        return []

    # CF scores
    candidates['cf_score'] = candidates['movieId'].apply(
        lambda mid: predict_rating(user_id, mid)
    )
    cf_min = candidates['cf_score'].min()
    cf_max = candidates['cf_score'].max()
    candidates['cf_norm'] = (candidates['cf_score']-cf_min)/(cf_max-cf_min+1e-9)

    # CB scores — anchor on movies user rated >= 4.0
    liked_ids = user_ratings[user_ratings['rating'] >= 4.0]['movieId'].tolist()

    def cb_score(movie_id):
        if movie_id not in mid_to_idx.index or not liked_ids:
            return 0.0
        idx      = mid_to_idx[movie_id]
        liked_i  = [mid_to_idx[m] for m in liked_ids if m in mid_to_idx.index]
        return float(np.mean(cosine_sim[idx, liked_i]))

    candidates['cb_score'] = candidates['movieId'].apply(cb_score)
    cb_min = candidates['cb_score'].min()
    cb_max = candidates['cb_score'].max()
    candidates['cb_norm'] = (candidates['cb_score']-cb_min)/(cb_max-cb_min+1e-9)

    # Blend
    candidates['hybrid_score'] = (
        ALPHA * candidates['cf_norm'] + (1-ALPHA) * candidates['cb_norm']
    )

    top = (
        candidates.nlargest(top_n, 'hybrid_score')
        .merge(links_df[['movieId','imdbId']], on='movieId', how='left')
    )

    results = []
    for _, row in top.iterrows():
        results.append({
            'movieId':      int(row['movieId']),
            'title':        row['title'],
            'genres':       row['genres'].replace('|', ', '),
            'hybrid_score': round(float(row['hybrid_score']), 4),
            'cf_score':     round(float(row['cf_norm']), 4),
            'cb_score':     round(float(row['cb_norm']), 4),
            'poster_url':   get_poster_url(row.get('imdbId')),})
    return results

def get_all_user_ids() -> list[int]:
    return sorted(ratings_df['userId'].unique().tolist())

Overwriting /content/drive/MyDrive/movie-recommender-clean/movie-recommender-clean/model/recommender.py


In [9]:
%%writefile /content/drive/MyDrive/movie-recommender-clean/movie-recommender-clean/api/main.py
# api/main.py
from fastapi import FastAPI, HTTPException, Query
from fastapi.middleware.cors import CORSMiddleware
import pandas as pd
import sys

BASE      = '/content/drive/MyDrive/movie-recommender-clean/movie-recommender-clean'
sys.path.insert(0, BASE) # Add BASE to sys.path

from model.recommender import get_hybrid_recommendations, get_all_user_ids

movies_df = pd.read_csv(f'{BASE}/data/movies.csv')

app = FastAPI(title='Movie Recommender API', version='1.0.0')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['GET'],
    allow_headers=['*'],
)

@app.get('/')
def root():
    return {'status': 'Movie Recommender API is running'}

@app.get('/recommend/{user_id}')
def recommend(user_id: int, top_n: int = Query(default=10, ge=1, le=50)):
    results = get_hybrid_recommendations(user_id, top_n=top_n)
    if not results:
        raise HTTPException(status_code=404, detail=f'No recs for user {user_id}')
    return {'user_id': user_id, 'total': len(results), 'recommendations': results}

@app.get('/users')
def list_users():
    return {'user_ids': get_all_user_ids()}

@app.get('/movies/search')
def search(q: str = Query(min_length=2)):
    matches = movies_df[movies_df['title'].str.contains(q,case=False,na=False)].head(20)
    return {'query':q,'results':matches[['movieId','title','genres']].to_dict(orient='records')}

In [10]:
import uvicorn, threading

def run(): uvicorn.run('api.main:app', host='0.0.0.0', port=8000)

t = threading.Thread(target=run, daemon=True)
t.start()
print('API running at http://localhost:8000')
print('Docs at      http://localhost:8000/docs')


API running at http://localhost:8000
Docs at      http://localhost:8000/docs
